# SAM3 Crop Visualization

This notebook allows you to visualize SAM3 crop regions for specific videos and frames.

**Purpose:** Validate that the SAM3 bounding boxes correctly identify the child of interest before cropping.


In [ ]:
import csv
import json
import subprocess
from pathlib import Path
from typing import Dict, List, Optional, Tuple
from datetime import datetime

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, HTML

import ipywidgets as widgets

%matplotlib inline
plt.rcParams['figure.figsize'] = [16, 10]

## Configuration


In [ ]:
# Paths
PARSED_CSV = Path('/orcd/data/satra/001/users/brukew/actreg/dataprep/rmm_sam3_parsed.csv')
ORIGINAL_VIDEO_BASE = Path('/orcd/data/satra/002/datasets/SAILS/Phase_III_Videos/Videos_from_external_standardized')
SAM3_OUTPUT_BASE = Path('/orcd/scratch/bcs/001/sensein/sails/rmm/rmm_sam_numbered')
MASK_CACHE_BASE = Path('/orcd/scratch/bcs/001/sensein/sails/cache_for_tracking')

# Rotation report (for handling rotated videos)
ROTATION_REPORT = Path('/orcd/data/satra/001/users/brukew/sailsprep/feature_processing/tracker/sam3/rotation_inconsistencies_report.csv')


## Load Data & Helper Functions


In [ ]:
def load_parsed_csv() -> List[Dict]:
    """Load the parsed SAM3 CSV with time intervals."""
    rows = []
    with open(PARSED_CSV, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader, start=1):
            row['row_idx'] = idx
            try:
                row['intervals'] = json.loads(row.get('child_sam3_ids', '[]'))
            except:
                row['intervals'] = []
            rows.append(row)
    return rows

def load_rotation_report() -> Dict[str, Dict]:
    """Load rotation metadata from the analysis report."""
    rotation_data = {}
    if ROTATION_REPORT.exists():
        with open(ROTATION_REPORT, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                filename = row.get('filename', '')
                stem = Path(filename).stem
                rotation_data[stem] = {
                    'rotation_metadata': row.get('rotation_metadata'),
                    'best_rotation': row.get('best_rotation'),
                    'rotation_issue': row.get('rotation_issue') == 'True',
                }
    return rotation_data

def get_video_paths(row_idx: int) -> Tuple[Optional[Path], Optional[Path], Optional[Path]]:
    """Get paths for original video, SAM3 output, and mask cache."""
    row = videos_data[row_idx - 1]
    source_file = row.get('SourceFile', '')
    filename = row.get('FileName', '')
    stem = Path(source_file).stem
    
    # Original video path
    print(stem)
    source_parts = source_file.split('/')
    if len(source_parts) > 1:
        original_path = ORIGINAL_VIDEO_BASE / '/'.join(source_parts[:-1]) / f"{stem}.mp4"
    else:
        original_path = ORIGINAL_VIDEO_BASE / source_parts[0] / f"{stem}.mp4"
    print(original_path)
    # SAM3 output path
    dir_num = row_idx + 1
    sam3_dir = SAM3_OUTPUT_BASE / f"{dir_num:03d}_{dir_num}"
    sam3_path = sam3_dir / f"{stem}_segmented.mp4"
    
    # Mask cache path - follows MaskCacheManager structure:
    # <cache_base>/masks/<video_basename>/<model>__prompt-<slug>.h5
    # where video_basename is the stem of the segmented video (e.g., "IMG_7467_segmented")
    video_basename = f"{stem}_segmented"
    mask_cache_path = MASK_CACHE_BASE / "masks" / video_basename / "facebook-sam3__prompt-person.h5"
    
    return (
        original_path if original_path.exists() else None,
        sam3_path if sam3_path.exists() else None,
        mask_cache_path if mask_cache_path.exists() else None,
    )

def get_video_rotation(video_path: Path) -> int:
    """Get rotation metadata from video file using ffprobe."""
    cmd = ["ffprobe", "-v", "error", "-select_streams", "v:0",
           "-show_entries", "stream_side_data=rotation", "-of", "csv=p=0", str(video_path)]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
        if result.stdout.strip():
            return int(float(result.stdout.strip()))
    except:
        pass
    return 0

def apply_rotation(frame: np.ndarray, rotation: int) -> np.ndarray:
    """Apply rotation to frame based on metadata."""
    if rotation == 0:
        return frame
    k = {90: 1, -90: 3, 180: 2, -180: 2, 270: 3, -270: 1}.get(rotation, 0)
    if k != 0:
        return np.rot90(frame, k)
    return frame

def get_frame_from_video(video_path: Path, frame_idx: int) -> Optional[np.ndarray]:
    """Extract a specific frame from a video file."""
    if not video_path or not video_path.exists():
        return None
    cap = cv2.VideoCapture(str(video_path))
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    cap.release()
    if ret:
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    return None

def get_video_info(video_path: Path) -> Dict:
    """Get video metadata."""
    cap = cv2.VideoCapture(str(video_path))
    info = {
        'width': int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
        'height': int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        'fps': cap.get(cv2.CAP_PROP_FPS),
        'frame_count': int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
    }
    cap.release()
    info['duration'] = info['frame_count'] / info['fps'] if info['fps'] > 0 else 0
    return info

# Load data
videos_data = load_parsed_csv()
rotation_data = load_rotation_report()
print(f"Loaded {len(videos_data)} videos")
print(f"Loaded rotation data for {len(rotation_data)} videos")


In [ ]:
def load_mask_cache(cache_path: Optional[Path],
                    video_basename: Optional[str] = None,
                    prompt: str = "person",
                    model_name: str = "facebook-sam3") -> Optional[Dict]:
    """
    Load SAM3 mask cache from HDF5 file.

    Resolution order:
        1) If cache_path is a file, use it.
        2) If cache_path is a directory, prefer `<model>__prompt-<slug>.h5`, else first .h5.
        3) If still unresolved, build the path using the MaskCacheManager layout:
           `<MASK_CACHE_BASE>/masks/<video_basename>/<model>__prompt-<slug>.h5`.
           Also tries `<video_basename>_segmented` and `<video_basename>_noviz`.

    Returns dict with frame_indices, obj_ids, boxes, scores, rles, attrs, and resolved path.
    """
    def _slugify(text: str) -> str:
        slug = "".join(ch.lower() if ch.isalnum() else "-" for ch in text.strip())
        while "--" in slug:
            slug = slug.replace("--", "-")
        return slug.strip("-") or "none"

    prompt_slug = _slugify(prompt)
    candidates = []

    # 1) Explicit path provided
    if cache_path:
        cache_path = Path(cache_path)
        if cache_path.is_file():
            candidates.append(cache_path)
        elif cache_path.is_dir():
            candidates.append(cache_path / f"{model_name}__prompt-{prompt_slug}.h5")
            candidates.extend(sorted(cache_path.glob("*.h5")))

    # 2) Paths derived from MaskCacheManager layout
    def add_from_basename(basename: str) -> None:
        base_dir = MASK_CACHE_BASE / "masks" / basename
        candidates.append(base_dir / f"{model_name}__prompt-{prompt_slug}.h5")
        candidates.extend(sorted(base_dir.glob("*.h5")))

    if video_basename:
        add_from_basename(video_basename)
        if not video_basename.endswith("_segmented"):
            add_from_basename(f"{video_basename}_segmented")
        if not video_basename.endswith("_noviz"):
            add_from_basename(f"{video_basename}_noviz")

    chosen = None
    for cand in candidates:
        if cand and cand.exists() and cand.is_file():
            chosen = cand
            break
    if chosen is None:
        return None

    try:
        with h5py.File(chosen, 'r') as f:
            frame_indices = []
            obj_ids = {}
            boxes = {}
            scores = {}
            rles = {}

            for key in f.keys():
                if not key.startswith('frame_'):
                    continue
                try:
                    frame_idx = int(key.split('_', 1)[1])
                except ValueError:
                    continue
                frame_indices.append(frame_idx)
                grp = f[key]
                obj_ids[frame_idx] = grp['obj_ids'][:]
                boxes[frame_idx] = grp['boxes'][:]
                scores[frame_idx] = grp['scores'][:]
                if 'rles' in grp:
                    rles[frame_idx] = [np.array(rle, dtype=np.uint8) for rle in grp['rles'][:]]

            if not frame_indices:
                print(f"⚠️  No frame groups found in cache {chosen}")
                return None

            attrs = {k: (int(v) if isinstance(v, np.integer) else v) for k, v in f.attrs.items()}

            return {
                'path': chosen,
                'frame_indices': np.array(sorted(frame_indices)),
                'obj_ids': obj_ids,
                'boxes': boxes,
                'scores': scores,
                'rles': rles,
                'attrs': attrs,
            }
    except Exception as e:
        print(f"Error loading cache {chosen}: {e}")
        return None

def rotate_boxes(boxes: np.ndarray, rotation: int, width: int, height: int) -> np.ndarray:
    """Rotate bounding boxes according to video rotation metadata."""
    if rotation == 0:
        return boxes
    rotated = boxes.copy()
    for i in range(len(boxes)):
        x1, y1, x2, y2 = boxes[i]
        if rotation in [-90, 270]:
            new_x1, new_y1 = y1, width - x2
            new_x2, new_y2 = y2, width - x1
            rotated[i] = [new_x1, new_y1, new_x2, new_y2]
        elif rotation in [90, -270]:
            new_x1, new_y1 = height - y2, x1
            new_x2, new_y2 = height - y1, x2
            rotated[i] = [new_x1, new_y1, new_x2, new_y2]
        elif rotation in [180, -180]:
            new_x1, new_y1 = width - x2, height - y2
            new_x2, new_y2 = width - x1, height - y1
            rotated[i] = [new_x1, new_y1, new_x2, new_y2]
    return rotated

def draw_boxes_on_frame(frame: np.ndarray, boxes: np.ndarray, obj_ids: np.ndarray, 
                        scores: np.ndarray, highlight_ids: Optional[List[int]] = None) -> np.ndarray:
    """Draw bounding boxes on frame. Green for highlighted IDs, red for others."""
    frame_out = frame.copy()
    for i, (box, obj_id, score) in enumerate(zip(boxes, obj_ids, scores)):
        x1, y1, x2, y2 = box.astype(int)
        if highlight_ids and int(obj_id) in highlight_ids:
            color = (0, 255, 0)
            thickness = 3
        else:
            color = (255, 0, 0)
            thickness = 2
        cv2.rectangle(frame_out, (x1, y1), (x2, y2), color, thickness)
        label = f"ID={int(obj_id)} ({score:.2f})"
        cv2.putText(frame_out, label, (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
    return frame_out


In [ ]:

# Save per-video rotation/cache metadata for reuse
from pathlib import Path

def save_video_metadata_cache(out_path: str = "video_metadata_cache.json", limit: int = None):
    """Collect rotation + size info per video and write to JSON for reuse.

    Args:
        out_path: Where to write the JSON (relative or absolute path).
        limit: Optional cap on number of rows to process (for quick tests).
    """
    records = []
    rows = videos_data[:limit] if limit else videos_data
    out_path = Path(out_path)

    for row in rows:
        row_idx = row.get('row_idx')
        filename = row.get('FileName', '')
        rec = {
            'row_idx': row_idx,
            'FileName': filename,
            'SourceFile': row.get('SourceFile', ''),
        }

        try:
            original_path, sam3_path, cache_path = get_video_paths(row_idx)
        except Exception as e:
            rec['error'] = f'path_error: {e}'
            records.append(rec)
            continue

        if original_path:
            rec['original_path'] = str(original_path)
            try:
                info = get_video_info(original_path)
                rec.update({
                    'width': info.get('width'),
                    'height': info.get('height'),
                    'fps': info.get('fps'),
                    'duration': info.get('duration'),
                })
                rec['rotation_meta'] = get_video_rotation(original_path)
            except Exception as e:
                rec['info_error'] = str(e)
        else:
            rec['error'] = 'missing original video'

        if cache_path:
            rec['mask_cache_path'] = str(cache_path)
            try:
                cache = load_mask_cache(cache_path, video_basename=None, prompt="person", model_name="facebook-sam3")
                if cache:
                    attrs = cache.get('attrs') or {}
                    rec['cache_width'] = attrs.get('width')
                    rec['cache_height'] = attrs.get('height')
                    rec['cache_stride'] = attrs.get('frame_stride')
                    rec['cache_frames'] = int(len(cache.get('frame_indices', []))) if cache.get('frame_indices') is not None else None
                else:
                    rec['cache_error'] = 'could not load cache'
            except Exception as e:
                rec['cache_error'] = str(e)
        records.append(rec)

    payload = {
        'generated_at': datetime.utcnow().isoformat() + 'Z',
        'count': len(records),
        'records': records,
    }
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(payload, indent=2))
    print(f"Saved {len(records)} entries to {out_path}")


## Main Visualization Function


In [ ]:
def visualize_frame(row_idx: int, frame_idx: int = 0, apply_rotation_fix: bool = True,
                    show_crop: bool = True, padding: int = 20, rotation_override: int = None,
                    autorotated_same_dims: bool = None):
    """
    Visualize a specific frame from a video with SAM3 detections.
    
    Args:
        row_idx: 1-based row index in the CSV
        frame_idx: Frame number to visualize
        apply_rotation_fix: Whether to apply rotation correction
        show_crop: Whether to show cropped region for child ID(s)
        padding: Padding around bounding box for crop
        rotation_override: If set, force this rotation (degrees) for boxes/frame (e.g., 90 or -90)
        autorotated_same_dims: For 180° cases where frame dims == cache dims, set True if the reader already applied rotation,
            False to force rotation, None to autodetect (default)
    """
    row = videos_data[row_idx - 1]
    filename = row.get('FileName', '')
    stem = Path(filename).stem
    intervals = row.get('intervals', [])
    
    print(f"Video: {filename}")
    print(f"SAM3 Label: {row.get('SAM3 label', '')}")
    print(f"SAM3 Notes: {row.get('SAM3 Notes', '')}")
    print(f"Child ID intervals: {intervals}")
    print()
    
    # Get paths
    original_path, sam3_path, cache_path = get_video_paths(row_idx)
    video_basename = sam3_path.stem if sam3_path else f"{stem}_segmented"
    
    if not original_path:
        print(f"❌ Original video not found")
        return
    
    # Get video info
    video_info = get_video_info(original_path)
    print(f"Video: {video_info['width']}x{video_info['height']}, {video_info['fps']:.2f} fps, {video_info['duration']:.2f}s")
    
    # Get rotation
    rotation_meta = get_video_rotation(original_path) * -1 if apply_rotation_fix else 0
    rotation_to_use = rotation_override if rotation_override is not None else rotation_meta
    if rotation_meta:
        print(f"Rotation metadata: {rotation_meta}°")
    if rotation_override is not None:
        print(f"Rotation override in use: {rotation_override}°")
    elif rotation_meta:
        print("Using rotation metadata for correction")
    
    # Load mask cache
    cache = load_mask_cache(cache_path, video_basename=video_basename, prompt="person", model_name="facebook-sam3")
    cache_width = cache_height = None
    if cache:
        cache_path_used = cache.get('path')
        cache_label = f"{cache_path_used}" if cache_path_used else "resolved"
        print(f"✓ Loaded mask cache ({cache_label}): {len(cache['frame_indices'])} frames")
        if cache.get('attrs'):
            attrs = cache['attrs']
            cache_width = attrs.get('width')
            cache_height = attrs.get('height')
            stride = attrs.get('frame_stride')
            print(f"   Cache dims: {cache_width}x{cache_height}, stride={stride}")
    else:
        print(f"⚠️  No mask cache found for {video_basename}")
    
    # Get frame
    frame = get_frame_from_video(original_path, frame_idx)
    if frame is None:
        print(f"❌ Could not read frame {frame_idx}")
        return
    
    # Determine dimensions and whether OpenCV already applied rotation
    frame_w, frame_h = frame.shape[1], frame.shape[0]
    base_width = cache_width or video_info['width']
    base_height = cache_height or video_info['height']
    print(f"Frame dims: {frame_w}x{frame_h} | Cache/base dims: {base_width}x{base_height}")

    auto_rotated_guess = bool(rotation_to_use) and frame_w == base_height and frame_h == base_width
    if autorotated_same_dims is True:
        auto_rotated = bool(rotation_to_use)
        print("Auto-rotated override: treating frame as already rotated (even though dims match)")
    elif autorotated_same_dims is False:
        auto_rotated = False
        print("Auto-rotated override: treating frame as NOT rotated")
    else:
        auto_rotated = auto_rotated_guess

    rotate_frame = bool(rotation_to_use) and apply_rotation_fix and not auto_rotated
    rotate_boxes_flag = bool(rotation_to_use) and apply_rotation_fix

    if rotate_frame:
        frame = apply_rotation(frame, rotation_to_use)
        frame_w, frame_h = frame.shape[1], frame.shape[0]
        print(f"Applied rotation to frame: {rotation_to_use}°")
    elif rotation_to_use and auto_rotated:
        print("Frame already matches rotated orientation; not rotating frame.")

    # Get child IDs for this frame time
    frame_time = frame_idx / video_info['fps']
    child_ids = []
    for iv in intervals:
        start = iv.get('start_sec', 0)
        end = iv.get('end_sec') or float('inf')
        if start <= frame_time < end:
            try:
                child_ids.append(int(iv['id']))
            except:
                pass
    
    print(f"Frame {frame_idx} (t={frame_time:.2f}s): Child IDs = {child_ids}")
    
    # Find boxes for this frame in cache
    boxes, obj_ids, scores = None, None, None
    if cache:
        frame_indices = cache['frame_indices']
        if len(frame_indices) > 0:
            # Find the closest cached frame to the requested frame
            closest_pos = np.argmin(np.abs(frame_indices - frame_idx))
            closest_frame_idx = frame_indices[closest_pos]
            
            # Access data using frame index as key (not list position)
            boxes = cache['boxes'][closest_frame_idx]
            obj_ids = cache['obj_ids'][closest_frame_idx]
            scores = cache['scores'][closest_frame_idx]
            print(f"Using cache frame {closest_frame_idx} (closest to {frame_idx})")
    
    # Identify matching detections
    matches = []
    if boxes is not None and obj_ids is not None and child_ids:
        matches = [i for i, obj in enumerate(obj_ids) if int(obj) in child_ids]
    if matches:
        matched_ids = [int(obj_ids[i]) for i in matches]
        print(f"Matched child IDs in frame: {matched_ids}")
    else:
        matched_ids = []
        print("No matching child IDs in this frame.")

    # Decide layout: always show the frame; optionally one combined crop
    if show_crop and matches:
        fig, axes = plt.subplots(1, 2, figsize=(18, 10))
        axes = list(axes)
    else:
        fig, axes = plt.subplots(1, 1, figsize=(12, 10))
        axes = [axes]
    
    # Draw boxes on frame
    if boxes is not None and len(boxes) > 0:
        boxes_rotated = boxes.copy()
        expected_w, expected_h = base_width, base_height

        if rotate_boxes_flag:
            boxes_rotated = rotate_boxes(boxes_rotated, rotation_to_use, base_width, base_height)
            if rotation_to_use in (-90, 90, -270, 270):
                expected_w, expected_h = base_height, base_width
            elif rotation_to_use in (-180, 180):
                expected_w, expected_h = base_width, base_height
        else:
            expected_w, expected_h = base_width, base_height

        # If frame size differs from expected (after rotation), scale to match display
        if expected_w and expected_h and (frame_w != expected_w or frame_h != expected_h):
            scale_x = frame_w / expected_w
            scale_y = frame_h / expected_h
            boxes_rotated[:, [0, 2]] *= scale_x
            boxes_rotated[:, [1, 3]] *= scale_y
            print(f"Scaling boxes from {expected_w}x{expected_h} -> frame {frame_w}x{frame_h}")

        frame_with_boxes = draw_boxes_on_frame(frame, boxes_rotated, obj_ids, scores, child_ids)
    else:
        frame_with_boxes = frame
        boxes_rotated = None
        print("⚠️  No detection boxes available")
    
    axes[0].imshow(frame_with_boxes)
    axes[0].set_title(f"Frame {frame_idx} (t={frame_time:.2f}s) - {filename}")
    axes[0].axis('off')
    
    # Show a single crop that covers all matching child IDs
    if show_crop and boxes_rotated is not None and matches:
        xs = []
        ys = []
        for det_idx in matches:
            x1, y1, x2, y2 = boxes_rotated[det_idx].astype(int)
            xs.extend([x1, x2])
            ys.extend([y1, y2])
        if xs and ys:
            x1, x2 = min(xs), max(xs)
            y1, y2 = min(ys), max(ys)
            h, w = frame.shape[:2]
            x1, y1 = max(0, x1 - padding), max(0, y1 - padding)
            x2, y2 = min(w, x2 + padding), min(h, y2 + padding)
            crop = frame[y1:y2, x1:x2]
            axes[1].imshow(crop)
            axes[1].set_title(f"Child IDs {matched_ids} crop ({x2-x1}x{y2-y1})")
            axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()


## Usage: List Videos


In [ ]:
# List videos with their indices
print("Videos with SAM3 annotations (first 30):")
print("-" * 90)
for row in videos_data[:30]:
    if row.get('parsed_ids'):
        rot_info = rotation_data.get(Path(row.get('FileName', '')).stem, {})
        rot_flag = "⚠️" if rot_info.get('rotation_issue') else "  "
        print(f"Row {row['row_idx']:3d} {rot_flag}: {row.get('FileName', '')[:45]:<45} IDs: {row.get('parsed_ids', '')}")


In [ ]:
# === CHANGE THESE VALUES ===
ROW_IDX = 45     # 1-based row index in the CSV
FRAME_IDX = 180  # Frame number to visualize

# visualize_frame(ROW_IDX, FRAME_IDX, apply_rotation_fix=True, show_crop=True, rotation_override=-180)
visualize_frame(ROW_IDX, FRAME_IDX, apply_rotation_fix=True, show_crop=True)


## Visualize a Specific Video

Change `ROW_IDX` and `FRAME_IDX` below to explore different videos and frames.


In [ ]:
# === CHANGE THESE VALUES ===
ROW_IDX = 321     # 1-based row index in the CSV
FRAME_IDX = 180  # Frame number to visualize

# visualize_frame(ROW_IDX, FRAME_IDX, apply_rotation_fix=True, show_crop=True, rotation_override=-180)
# visualize_frame(ROW_IDX, FRAME_IDX, apply_rotation_fix=True, show_crop=True, autorotated_same_dims=True)



## Compare Multiple Frames


In [ ]:
# Visualize multiple frames from a video
ROW_IDX = 186
FRAMES = [0, 100, 400, 800, 100]  # Frame numbers to visualize

# for frame_idx in FRAMES:
#     print("=" * 80)
#     visualize_frame(ROW_IDX, frame_idx, apply_rotation_fix=True, show_crop=True, autorotated_same_dims=True)


In [ ]:
# Visualize multiple frames from a video
ROW_IDX = 45
FRAMES = [390, 400, 410, 405, 420]  # Frame numbers to visualize

for frame_idx in FRAMES:
    print("=" * 80)
    visualize_frame(ROW_IDX, frame_idx, apply_rotation_fix=True, show_crop=True, autorotated_same_dims=True)


In [ ]:
# Find videos with rotation metadata
rotation_videos = []
for row in videos_data:
    stem = Path(row.get('FileName', '')).stem
    rot_info = rotation_data.get(stem, {})
    if rot_info.get('rotation_issue'):
        rotation_videos.append({
            'row_idx': row['row_idx'],
            'filename': row.get('FileName', ''),
            'rotation': rot_info.get('rotation_metadata'),
            'child_ids': row.get('parsed_ids', ''),
        })

print(f"Videos with rotation issues: {len(rotation_videos)}")
print()
for v in rotation_videos[:15]:
    print(f"Row {v['row_idx']:3d}: {v['filename'][:40]:<40} rot={str(v['rotation']):>5}° IDs: {v['child_ids']}")


## Pose Estimation on Crops
Use SAM3 boxes to crop the child region and run a YOLO pose model. Adjust the config cell below and call `pose_on_crops(...)`.

In [ ]:

try:
    from ultralytics import YOLO
except ImportError as e:
    raise ImportError("Install ultralytics to run YOLO pose (pip install ultralytics)") from e

from sam3_crops_utils import load_parsed_csv, load_rotation_report, prepare_frame_and_boxes

# Config
POSE_MODEL = "yolo11x-pose.pt"  # or yolo11x-pose.pt, etc.
POSE_DEVICE = None  # set to "cuda" if available
POSE_PAD = 20
POSE_CONF = 0.25

# Load shared data once
_pose_videos_data = load_parsed_csv()
_pose_rotation_data = load_rotation_report()
_pose_model = YOLO(POSE_MODEL)
if POSE_DEVICE:
    _pose_model.to(POSE_DEVICE)


def build_pose_input(frame_rgb: np.ndarray, mask_bool: np.ndarray, blur_ksize: int = 51, falloff_px: float = 120.0) -> np.ndarray:
    """Blend original frame with a blurred background based on distance to the mask."""
    if blur_ksize % 2 == 0:
        blur_ksize += 1
    background = (~mask_bool).astype(np.uint8)
    heavy_blur = cv2.GaussianBlur(frame_rgb, (blur_ksize, blur_ksize), sigmaX=0)
    dist = cv2.distanceTransform(background, cv2.DIST_L2, 3)
    if falloff_px <= 0:
        weight = np.zeros_like(dist, dtype=np.float32)
    else:
        weight = np.exp(-dist / float(falloff_px))
    weight = weight[..., None].astype(np.float32)
    blended = heavy_blur.astype(np.float32) * (1.0 - weight) + frame_rgb.astype(np.float32) * weight
    blended[mask_bool] = frame_rgb[mask_bool]
    return np.clip(blended, 0, 255).astype(np.uint8)


def build_pose_input_black(frame_rgb: np.ndarray, mask_bool: np.ndarray, buffer_px: float = 30.0) -> np.ndarray:
    """Keep subject and a distance buffer, set all other pixels to black."""
    background = (~mask_bool).astype(np.uint8)
    if buffer_px <= 0:
        keep = mask_bool
    else:
        dist = cv2.distanceTransform(background, cv2.DIST_L2, 3)
        keep = (dist <= float(buffer_px)) | mask_bool
    output = np.zeros_like(frame_rgb)
    output[keep] = frame_rgb[keep]
    return output


def pose_on_crops(
    row_idx: int,
    frame_idx: int,
    target_ids=None,
    pad: int = POSE_PAD,
    conf: float = POSE_CONF,
    device: str | None = POSE_DEVICE,
    model=None,
    input_mode: str = "blur",  # blur | black | raw
    buffer_px: int = 30,
    blur_ksize: int = 51,
    falloff_px: float = 120.0,
):
    """Run YOLO pose on target IDs; skip if targets not present. If only targets are present, run on the full frame."""
    model = model or _pose_model
    data = prepare_frame_and_boxes(
        row_idx=row_idx,
        frame_idx=frame_idx,
        videos_data=_pose_videos_data,
        rotation_data=_pose_rotation_data,
        return_masks=True,
    )
    if data is None:
        print("Could not load frame/boxes for row", row_idx)
        return

    frame = data["frame"]
    boxes = data["boxes"]
    obj_ids = data["obj_ids"]
    scores = data["scores"]
    masks = data.get("masks")
    child_ids = data.get("child_ids") or []

    if boxes is None or obj_ids is None or scores is None:
        print("No boxes available for this frame")
        return

    target_set = set(int(t) for t in (target_ids or child_ids) if t is not None)
    if not target_set:
        print("No target IDs configured for this frame/video.")
        return

    keep = [i for i, oid in enumerate(obj_ids) if int(oid) in target_set]
    if not keep:
        print("Target IDs not present in this frame; skipping.")
        return

    boxes = boxes[keep]
    obj_ids = obj_ids[keep]
    scores = scores[keep]
    if masks is not None and len(masks) == len(data.get("obj_ids", [])):
        masks = [masks[i] for i in keep]
    else:
        masks = None

    import matplotlib.pyplot as plt

    full_frame_only_targets = len(keep) == len(data.get("obj_ids", []))
    if full_frame_only_targets:
        # If we have masks, use the first target mask for black/blur; otherwise raw frame
        # if masks is not None and len(masks) > 0:
        #     m0 = masks[0]
        #     if input_mode == "black":
        #         pose_frame = build_pose_input_black(frame, m0, buffer_px)
        #     elif input_mode == "blur":
        #         pose_frame = build_pose_input(frame, m0, blur_ksize=blur_ksize, falloff_px=falloff_px)
        #     else:
        #         pose_frame = frame
        # else:
        pose_frame = frame
        res = model.predict(cv2.cvtColor(pose_frame, cv2.COLOR_RGB2BGR), conf=conf, verbose=False, device=device)[0]
        annotated_bgr = res.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(7, 7))
        plt.imshow(annotated_rgb)
        plt.title("All targets present; full frame pose")
        plt.axis("off")
        plt.show()
        return

    n = len(boxes)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, box, oid, score, idx in zip(axes, boxes, obj_ids, scores, range(n)):
        h, w = frame.shape[:2]
        x1, y1, x2, y2 = box.astype(int)
        x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
        x2, y2 = min(w, x2 + pad), min(h, y2 + pad)

        mask_bool = None
        if masks is not None and idx < len(masks):
            mask_bool = masks[idx]

        if mask_bool is not None:
            if input_mode == "black":
                pose_frame = build_pose_input_black(frame, mask_bool, buffer_px)
            elif input_mode == "raw":
                pose_frame = frame
            else:
                pose_frame = build_pose_input(frame, mask_bool, blur_ksize=blur_ksize, falloff_px=falloff_px)
        else:
            pose_frame = frame

        crop_rgb = pose_frame[y1:y2, x1:x2]
        if crop_rgb.size == 0:
            ax.set_title(f"id {int(oid)} (empty crop)")
            ax.axis("off")
            continue

        crop_bgr = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR)
        res = model.predict(crop_bgr, conf=conf, verbose=False, device=device)[0]
        annotated_bgr = res.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

        ax.imshow(annotated_rgb)
        ax.set_title(f"id {int(oid)} | det score {float(score):.2f}")
        ax.axis("off")

    plt.show()

In [ ]:
for frame in FRAMES:
    pose_on_crops(row_idx=316, frame_idx=frame, target_ids=None, pad=20, conf=0.25, input_mode="black", buffer_px=30)

## Visualize All Masks in a Frame
Quick helper to overlay all SAM3 masks for a given row/frame.

In [ ]:

from sam3_crops_utils import load_parsed_csv, load_rotation_report, prepare_frame_and_boxes

# Load once
_vis_videos_data = load_parsed_csv()
_vis_rotation_data = load_rotation_report()


def visualize_masks_for_frame(row_idx: int, frame_idx: int, alpha: float = 0.4, figsize=(12, 6), target_ids=None):
    """Show original frame with aligned SAM3 masks/boxes for target IDs (or all if None)."""
    data = prepare_frame_and_boxes(
        row_idx=row_idx,
        frame_idx=frame_idx,
        videos_data=_vis_videos_data,
        rotation_data=_vis_rotation_data,
        return_masks=True,
        target_obj_ids=target_ids,
    )
    if data is None:
        print(f"Could not load frame/boxes for row {row_idx} frame {frame_idx}")
        return

    frame = data["frame"]
    boxes = data.get("boxes")
    obj_ids = data.get("obj_ids")
    scores = data.get("scores")
    masks = data.get("masks") or []

    if boxes is None or obj_ids is None or scores is None:
        print("No target IDs present in this frame.")
        return

    import matplotlib.pyplot as plt
    import numpy as np
    import cv2

    # Frame with boxes (targets only)
    boxed = frame.copy()
    for box, oid, score in zip(boxes, obj_ids, scores):
        x1, y1, x2, y2 = box.astype(int)
        color = (0, 255, 0)
        cv2.rectangle(boxed, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            boxed,
            f"id={int(oid)} ({float(score):.2f})",
            (x1, max(y1 - 5, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
            lineType=cv2.LINE_AA,
        )

    # Frame with masks (targets only)
    blended = frame.copy()
    rng = np.random.default_rng(123)
    for m in masks:
        color = rng.integers(0, 255, size=3, dtype=np.uint8)
        mask_bool = m.astype(bool)
        blended[mask_bool] = (alpha * color + (1 - alpha) * blended[mask_bool]).astype(np.uint8)

    fig, axes = plt.subplots(1, 3, figsize=figsize if isinstance(figsize, tuple) else (18, 6))
    axes[0].imshow(frame)
    axes[0].set_title(f"Frame {frame_idx}")
    axes[0].axis("off")

    axes[1].imshow(boxed)
    axes[1].set_title("Boxes (targets)")
    axes[1].axis("off")

    axes[2].imshow(blended)
    axes[2].set_title("Masks (targets)")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# Example:
visualize_masks_for_frame(row_idx=45, frame_idx=0, target_ids=[123])


## Full-video pose export for target IDs
Run YOLO pose across a full video for the target IDs, using SAM3 masks to mask/crop. If the frame only contains target IDs, pose runs on the full frame; otherwise it crops per target and maps keypoints back to frame coords. Outputs an H.264 MP4.

In [ ]:

from pathlib import Path
import subprocess
from tqdm import tqdm
from pycocotools import mask as maskUtils


def build_ffmpeg_writer(output_path: Path, width: int, height: int, fps: float, loglevel: str = "error"):
    cmd = [
        "ffmpeg", "-loglevel", loglevel, "-y",
        "-f", "rawvideo", "-vcodec", "rawvideo", "-pix_fmt", "bgr24",
        "-s", f"{width}x{height}", "-r", str(fps if fps > 0 else 30), "-i", "-",
        "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p", str(output_path),
    ]
    return subprocess.Popen(cmd, stdin=subprocess.PIPE)


def draw_keypoints(frame_bgr, res, color, offset=(0, 0)):
    if res.keypoints is None:
        return
    pts = res.keypoints.xy if hasattr(res.keypoints, "xy") else res.keypoints.data
    if pts is None:
        return
    pts = pts.cpu().numpy()
    ox, oy = offset
    for det_pts in pts:
        for x, y in det_pts:
            cv2.circle(frame_bgr, (int(x) + ox, int(y) + oy), 3, color, -1, lineType=cv2.LINE_AA)


def decode_masks_for_frame(cache, frame_idx: int, rotation: int, frame_w: int, frame_h: int, base_w: int, base_h: int):
    masks_out = []
    if cache is None or 'rles' not in cache or frame_idx not in cache['rles']:
        return masks_out
    rles = cache['rles'][frame_idx]
    for rle in rles:
        rle_dict = {"counts": rle.tobytes(), "size": [base_h, base_w]}
        m = maskUtils.decode(rle_dict)
        if m.ndim == 3:
            m = m[:, :, 0]
        if rotation:
            m = apply_rotation(m, rotation)
        if m.shape != (frame_h, frame_w):
            m = cv2.resize(m.astype(np.uint8), (frame_w, frame_h), interpolation=cv2.INTER_NEAREST)
        masks_out.append(m.astype(bool))
    return masks_out


def run_pose_video_targets(
    row_idx: int,
    output_path: str | Path,
    target_ids=None,
    input_mode: str = "blur",  # blur | black | raw
    buffer_px: int = 30,
    blur_ksize: int = 61,
    falloff_px: float = 120.0,
    pad: int = 20,
    conf: float = 0.25,
    device: str | None = POSE_DEVICE,
    model=None,
    max_frames: int | None = None,
    ffmpeg_loglevel: str = "error",
):
    model = model or _pose_model
    row = videos_data[row_idx - 1]
    filename = row.get('FileName', '')
    intervals = row.get('intervals', [])

    original_path, sam3_path, cache_path = get_video_paths(row_idx)
    if not original_path:
        print("Missing original video")
        return

    cache = load_mask_cache(cache_path, video_basename=sam3_path.stem if sam3_path else None, prompt="person", model_name="facebook-sam3")
    cache_width = cache_height = None
    if cache and cache.get('attrs'):
        cache_width = cache['attrs'].get('width')
        cache_height = cache['attrs'].get('height')
    base_w = cache_width or 0
    base_h = cache_height or 0

    cap = cv2.VideoCapture(str(original_path))
    if not cap.isOpened():
        print("Could not open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    total = min(frame_count, max_frames) if max_frames else frame_count

    rotation_meta = get_video_rotation(original_path) * -1 if True else 0

    # Child IDs per frame from intervals
    def child_ids_for_time(t):
        ids = []
        for iv in intervals:
            start = iv.get('start_sec', 0)
            end = iv.get('end_sec') or float('inf')
            if start <= t < end:
                try:
                    ids.append(int(iv['id']))
                except Exception:
                    pass
        return ids

    out_path = Path(output_path)
    writer = None

    for idx in tqdm(range(total), desc=f"row{row_idx}", leave=False):
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame_w, frame_h = frame_rgb.shape[1], frame_rgb.shape[0]

        base_width = cache_width or frame_w
        base_height = cache_height or frame_h
        auto_rotated_guess = bool(rotation_meta) and frame_w == base_height and frame_h == base_width
        rotate_frame = bool(rotation_meta) and not auto_rotated_guess
        rotate_boxes_flag = bool(rotation_meta)

        if rotate_frame:
            frame_rgb = apply_rotation(frame_rgb, rotation_meta)
            frame_w, frame_h = frame_rgb.shape[1], frame_rgb.shape[0]

        # Grab detections from cache for this frame
        boxes = obj_ids = scores = None
        masks = None
        if cache and idx in cache.get('boxes', {}):
            boxes = cache['boxes'][idx]
            obj_ids = cache['obj_ids'][idx]
            scores = cache['scores'][idx]
            masks = decode_masks_for_frame(cache, idx, rotation_meta if rotate_boxes_flag else 0, frame_w, frame_h, base_width, base_height)

        if boxes is None or obj_ids is None:
            if writer is None:
                writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
            writer.stdin.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).tobytes())
            continue

        boxes_rot = boxes.copy()
        expected_w, expected_h = base_width, base_height
        if rotate_boxes_flag and rotation_meta:
            boxes_rot = rotate_boxes(boxes_rot, rotation_meta, base_width, base_height)
            if rotation_meta in (-90, 90, -270, 270):
                expected_w, expected_h = base_height, base_width
            elif rotation_meta in (-180, 180):
                expected_w, expected_h = base_width, base_height
        if expected_w and expected_h and (frame_w != expected_w or frame_h != expected_h):
            scale_x = frame_w / expected_w
            scale_y = frame_h / expected_h
            boxes_rot[:, [0, 2]] *= scale_x
            boxes_rot[:, [1, 3]] *= scale_y
            if masks:
                masks = [cv2.resize(m.astype(np.uint8), (frame_w, frame_h), interpolation=cv2.INTER_NEAREST).astype(bool) for m in masks]

        frame_time = idx / fps if fps else 0.0
        target_set = set(int(t) for t in (target_ids or child_ids_for_time(frame_time)) if t is not None)
        keep = [i for i, oid in enumerate(obj_ids) if int(oid) in target_set]
        if not keep:
            if writer is None:
                writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
            writer.stdin.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR).tobytes())
            continue

        only_targets = len(keep) == len(obj_ids)
        frame_vis = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)

        if only_targets:
            combined_mask = None
            if masks:
                combined_mask = np.any([masks[i] for i in keep], axis=0)
            # if combined_mask is not None:
            #     if input_mode == "black":
            #         pose_frame = build_pose_input_black(frame_rgb, combined_mask, buffer_px)
            #     elif input_mode == "blur":
            #         pose_frame = build_pose_input(frame_rgb, combined_mask, blur_ksize=blur_ksize, falloff_px=falloff_px)
            #     else:
            #         pose_frame = frame_rgb
            # else:
            pose_frame = frame_rgb
            res = model.predict(cv2.cvtColor(pose_frame, cv2.COLOR_RGB2BGR), conf=conf, verbose=False, device=device)[0]
            color = (0, 255, 0)
            draw_keypoints(frame_vis, res, color, offset=(0, 0))
        else:
            for j, ki in enumerate(keep):
                box = boxes_rot[ki]
                h, w = frame_rgb.shape[:2]
                x1, y1, x2, y2 = box.astype(int)
                x1, y1 = max(0, x1 - pad), max(0, y1 - pad)
                x2, y2 = min(w, x2 + pad), min(h, y2 + pad)
                if x2 <= x1 or y2 <= y1:
                    continue
                mask_bool = None
                if masks and ki < len(masks):
                    mask_bool = masks[ki]
                if mask_bool is not None:
                    if input_mode == "black":
                        pose_frame = build_pose_input_black(frame_rgb, mask_bool, buffer_px)
                    elif input_mode == "raw":
                        pose_frame = frame_rgb
                    else:
                        pose_frame = build_pose_input(frame_rgb, mask_bool, blur_ksize=blur_ksize, falloff_px=falloff_px)
                else:
                    pose_frame = frame_rgb
                crop_rgb = pose_frame[y1:y2, x1:x2]
                if crop_rgb.size == 0:
                    continue
                crop_bgr = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2BGR)
                res = model.predict(crop_bgr, conf=conf, verbose=False, device=device)[0]
                color = (
                    (int(obj_ids[ki]) * 50) % 255,
                    (int(obj_ids[ki]) * 100) % 255,
                    (int(obj_ids[ki]) * 150) % 255,
                )
                draw_keypoints(frame_vis, res, color, offset=(x1, y1))

        if writer is None:
            writer = build_ffmpeg_writer(out_path, frame_w, frame_h, fps, loglevel=ffmpeg_loglevel)
        writer.stdin.write(frame_vis.astype(np.uint8).tobytes())

    if writer is not None:
        writer.stdin.close()
        writer.wait()
    cap.release()
    print(f"Saved pose video to {out_path}")

# Example usage:
run_pose_video_targets(row_idx=45, output_path="row45_pose.mp4", target_ids=None, input_mode="black", buffer_px=30, conf=0.25)
